# PaperLens
RAG pipeline for question answering over research papers with citations.

In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters langchain-core sentence-transformers faiss-cpu pymupdf groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Groq API key: ')

Groq API key: ··········


In [3]:
from google.colab import files
uploaded = files.upload()
pdf_paths = list(uploaded.keys())

Saving SAMPLE PPR.pdf to SAMPLE PPR.pdf


In [4]:
import fitz
from dataclasses import dataclass

@dataclass
class PageContent:
    doc_name: str
    page_number: int
    text: str

def extract_pages(pdf_path):
    doc_name = pdf_path.split('/')[-1]
    pages = []
    with fitz.open(pdf_path) as doc:
        for i, page in enumerate(doc):
            text = page.get_text('text').strip()
            if text:
                pages.append(PageContent(doc_name, i + 1, text))
    return pages

all_pages = []
for p in pdf_paths:
    all_pages.extend(extract_pages(p))

len(all_pages)

11

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', '']
)

documents = []
for page in all_pages:
    for piece in splitter.split_text(page.text):
        documents.append(Document(
            page_content=piece,
            metadata={'doc_name': page.doc_name, 'page_number': page.page_number}
        ))

len(documents)

53

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    encode_kwargs={'normalize_embeddings': True}
)

vector_store = FAISS.from_documents(documents, embeddings)
vector_store.index.ntotal

/tmp/ipykernel_1091/1839299023.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

53

In [7]:
from groq import Groq

client = Groq(api_key=os.environ['GROQ_API_KEY'])

PROMPT_TEMPLATE = '''You are PaperLens, an assistant that answers questions strictly using the provided research paper excerpts.
Only use information found in the CONTEXT. If the answer isn't there, say so honestly. Be concise.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:'''

def ask(question, k=4):
    results = vector_store.similarity_search_with_relevance_scores(question, k=k)
    context = '\n\n'.join(
        f"[Source {i+1} | {doc.metadata['doc_name']} | Page {doc.metadata['page_number']}]\n{doc.page_content}"
        for i, (doc, score) in enumerate(results)
    )
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2,
        max_tokens=800,
    )
    answer = response.choices[0].message.content.strip()

    print(answer)
    print()
    for doc, score in results:
        print(f"{doc.metadata['doc_name']} (page {doc.metadata['page_number']}, relevance {score:.3f})")
    return answer, results

In [8]:
_ = ask('What problem does this paper try to solve, and what method do they propose?')

/tmp/ipykernel_1091/3437783763.py:16: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='a59367e8-effc-49f8-9a31-7ed85a5fbd8e', metadata={'doc_name': 'SAMPLE PPR.pdf', 'page_number': 7}, page_content='(3)\nThis corresponds to increasing the learning rate linearly for the ﬁrst warmup_steps training steps,\nand decreasing it thereafter proportionally to the inverse square root of the step number. We used\nwarmup_steps = 4000.\n5.4\nRegularization\nWe employ three types of regularization during training:\nResidual Dropout\nWe apply dropout [27] to the output of each sub-layer, before it is added to the\nsub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the\npositional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7'), np.float32(-0.13292825)), (Document(id='74d77a8e-67df-4726-bda9-681074c66971', metadata={'doc_name': 'SAMPLE PPR.pdf', 'page_number': 4}, page_content

Unfortunately, the provided context does not explicitly state the problem the paper tries to solve or the method they propose. The context appears to be discussing various aspects of a model, including learning rate, regularization, and attention mechanisms, but it does not provide a clear problem statement or a concise description of the proposed method.

SAMPLE PPR.pdf (page 7, relevance -0.133)
SAMPLE PPR.pdf (page 4, relevance -0.136)
SAMPLE PPR.pdf (page 10, relevance -0.140)
SAMPLE PPR.pdf (page 2, relevance -0.145)
